# Fine-Tuning XLM-R: Sentimiento (ES + PT)

**Cero datos sintéticos** — solo datos reales:
- **ES:** Amazon reviews (160k, positive/negative balanceado)
- **PT:** Corpus e-commerce (111k, positive/negative)

**Modelo:** `xlm-roberta-base` → clasificación binaria positive/negative

**Checkpoint automático:** HuggingFace Trainer guarda en Drive cada epoch.
Si Colab se cierra, solo montás Drive y continuás desde el último checkpoint.

In [ ]:
# @title 1. Montar Google Drive
from google.colab import drive
import os, sys, torch, pandas as pd, numpy as np
from IPython.display import display, HTML

drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/ConversaAI'
os.makedirs(f'{DRIVE_DIR}/data', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/models', exist_ok=True)

print(f'Drive montado en: {DRIVE_DIR}')
print('\n⚠️ Subí los datasets CSV a:')
print(f'   {DRIVE_DIR}/data/')
print('   1. spanish_amazon_reviews.csv (ES, 160k)')
print('   2. corpus_ecommerce.csv (PT, 111k)')

# Verificar
!ls -lh "$DRIVE_DIR/data/" 2>/dev/null || echo '(vacío)'

In [ ]:
# @title 2. Verificar GPU
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

!nvidia-smi

In [ ]:
# @title 3. Instalar dependencias
!pip install -q transformers datasets accelerate scikit-learn pandas numpy

In [ ]:
# @title 4. Cargar datos desde Drive
from sklearn.model_selection import train_test_split

SAMPLE_SIZE = None  # None = usar todo, o poné 5000 para prueba rápida
RESUME = False  # True para retomar entrenamiento desde checkpoint

def load_sentiment_data():
    datasets = []

    es_path = f'{DRIVE_DIR}/data/spanish_amazon_reviews.csv'
    if not os.path.exists(es_path):
        # Fallback a local
        es_path = 'data/spanish_amazon_reviews.csv'

    es = pd.read_csv(es_path)
    es = es[es['sentiment'].isin(['positive', 'negative'])].copy()
    es['lang'] = 'es'
    print(f"ES (Amazon): {len(es)} filas")
    print(f"   positive: {(es['sentiment']=='positive').sum()}")
    print(f"   negative: {(es['sentiment']=='negative').sum()}")
    datasets.append(es)

    pt_path = f'{DRIVE_DIR}/data/corpus_ecommerce.csv'
    if not os.path.exists(pt_path):
        pt_path = 'data/corpus_ecommerce.csv'

    pt = pd.read_csv(pt_path)
    pt = pt[pt['lang'] == 'pt'].copy()
    pt = pt[pt['sentiment'].isin(['positive', 'negative'])].copy()
    pt = pt[pt['role'] == 'customer'].copy()
    pt['lang'] = 'pt'
    print(f"\nPT (Corpus e-commerce): {len(pt)} filas")
    print(f"   positive: {(pt['sentiment']=='positive').sum()}")
    print(f"   negative: {(pt['sentiment']=='negative').sum()}")
    datasets.append(pt)

    df = pd.concat(datasets, ignore_index=True)
    label_map = {'negative': 0, 'positive': 1}
    df['label'] = df['sentiment'].map(label_map)

    if SAMPLE_SIZE:
        sampled = []
        for lang in ['es', 'pt']:
            sub = df[df['lang'] == lang]
            pos = sub[sub['sentiment'] == 'positive']
            neg = sub[sub['sentiment'] == 'negative']
            n = SAMPLE_SIZE // 2
            sampled.append(pd.concat([
                pos.sample(n=min(len(pos), n), random_state=42),
                neg.sample(n=min(len(neg), n), random_state=42)
            ]))
        df = pd.concat(sampled, ignore_index=True)
        print(f"\n⚠️ Sample activo: {len(df)} total")

    print(f"\n{'='*60}")
    print(f"TOTAL: {len(df)} muestras")
    print(f"{'='*60}")
    print(df.groupby('lang')['sentiment'].value_counts().unstack(fill_value=0).to_string())
    return df

df = load_sentiment_data()

In [ ]:
# @title 5. Split + Tokenizer
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = "xlm-roberta-base"
MAX_LENGTH = 128
BATCH_SIZE = 32

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

df['stratify_key'] = df['lang'] + '_' + df['sentiment']
train_df, temp_df = train_test_split(
    df, test_size=0.3, random_state=42, stratify=df['stratify_key']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df['stratify_key']
)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"\nTrain balance:")
print(train_df.groupby('lang')['sentiment'].value_counts().unstack(fill_value=0))

def tok(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=MAX_LENGTH)

train_ds = Dataset.from_pandas(train_df[['text', 'label']]).map(tok, batched=True).remove_columns(['text']).set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
val_ds = Dataset.from_pandas(val_df[['text', 'label']]).map(tok, batched=True).remove_columns(['text']).set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_ds = Dataset.from_pandas(test_df[['text', 'label']]).map(tok, batched=True).remove_columns(['text']).set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

print(f"\nDatasets listos: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")

In [ ]:
# @title 6. Entrenar (con checkpoints en Drive y resume)
from transformers import (
    AutoModelForSequenceClassification,
    Trainer, TrainingArguments, EarlyStoppingCallback
)

# Output en Drive para persistencia
OUTPUT_DIR = f'{DRIVE_DIR}/models/xlmr-sentiment'
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5

# Detectar si hay checkpoint previo
checkpoint = None
if RESUME and os.path.exists(OUTPUT_DIR) and os.path.exists(f'{OUTPUT_DIR}/config.json'):
    checkpoints = [d for d in os.listdir(OUTPUT_DIR) if d.startswith('checkpoint-')]
    if checkpoints:
        checkpoint = os.path.join(OUTPUT_DIR, sorted(checkpoints)[-1])
        print(f"⏩ Checkpoint encontrado: {checkpoint}")
        print("   Se retomará desde allí.")

print("Cargando XLM-RoBERTa...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "negative", 1: "positive"},
    label2id={"negative": 0, "positive": 1}
)

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.argmax(preds, axis=1)
    return {'accuracy': (preds == labels).mean()}

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    greater_is_better=True,
    fp16=True,
    logging_steps=100,
    report_to='none',
    log_level='error',
    dataloader_num_workers=2
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print(f"\n{'='*60}")
print(f"Iniciando entrenamiento...")
print(f"  Train: {len(train_ds)} | Val: {len(val_ds)}")
print(f"  Epochs: {NUM_EPOCHS} | Batch: {BATCH_SIZE} | LR: {LEARNING_RATE}")
print(f"  Output: {OUTPUT_DIR} (en Drive!)")
print(f"  Checkpoint previo: {checkpoint or 'no'}")
print(f"{'='*60}")

trainer.train(resume_from_checkpoint=checkpoint)

In [ ]:
# @title 7. Evaluar
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

results = trainer.predict(test_ds)
preds = np.argmax(results.predictions, axis=1)

print(f"{'='*60}")
print(f"RESULTADOS - SENTIMIENTO")
print(f"{'='*60}")

overall_acc = accuracy_score(test_df['label'], preds)
overall_f1 = f1_score(test_df['label'], preds, average='weighted')
print(f"\n🌍 Global: Accuracy: {overall_acc*100:.2f}% | F1: {overall_f1*100:.2f}%")

for lang in ['es', 'pt']:
    mask = test_df['lang'] == lang
    if mask.sum() == 0:
        continue
    lang_labels = test_df[mask]['label'].values
    lang_preds = preds[mask.values]
    acc = accuracy_score(lang_labels, lang_preds)
    f1 = f1_score(lang_labels, lang_preds, average='weighted')
    print(f"\n{'='*40}")
    print(f"  {lang.upper()} — Accuracy: {acc*100:.2f}% | F1: {f1*100:.2f}%")
    print(f"{'='*40}")
    print(classification_report(lang_labels, lang_preds, target_names=['negative', 'positive']))

    cm = confusion_matrix(lang_labels, lang_preds)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['negative', 'positive'], yticklabels=['negative', 'positive'])
    plt.title(f'Confusion Matrix - {lang.upper()}')
    plt.ylabel('True')
    plt.xlabel('Predicted')
    plt.show()

In [ ]:
# @title 8. Guardar y copiar a local para descarga
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Copia local para zip/descarga
!cp -r "$OUTPUT_DIR" ./xlmr-sentiment/
!zip -r xlmr-sentiment-v3.zip xlmr-sentiment/

print(f"✅ Modelo guardado en Drive: {OUTPUT_DIR}")
print(f"   Persiste aunque se cierre Colab.")
print(f"\n✅ Zip listo para descargar: xlmr-sentiment-v3.zip")
print(f"   Tamaño: {os.path.getsize('xlmr-sentiment-v3.zip') / 1e6:.1f} MB")

from google.colab import files
files.download('xlmr-sentiment-v3.zip')

---
## Resumen

| Concepto | Detalle |
|----------|---------|
| Modelo | XLM-RoBERTa Base |
| Datos ES | Amazon reviews (160k) |
| Datos PT | Corpus e-commerce (111k) |
| Total | ~271k muestras reales |
| Checkpoints | `{DRIVE_DIR}/models/xlmr-sentiment/` |

**Cero datos sintéticos. Persistido en Drive.**

Para retomar: montá Drive, poné `RESUME = True` en celda 4, y ejecutá todo.